In [1]:

import pandas as pd
import plotly.express as px
import calendar
from datetime import datetime
from datetime import timedelta
from cronsim import CronSim

from datetime import date

today = date.today()
import argparse
import json
#import PySimpleGUI as sg
#import PySimpleGUIWeb as sg
import pathlib
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt
import textwrap
import re
import webbrowser
import subprocess
import collections
import pyglet,tkinter
from pyglet import font
import os
import subprocess
import requests
import gspread
from oauth2client.service_account import ServiceAccountCredentials

# import OpenGL
# from OpenGL import GLU
font.add_file('/etc/fonts/fonts/CENTAUR.TTF')
font='Courier 10 bold '

In [7]:
crons = []
def find_all(a_str, sub):
    start = 0
    while True:
        start = a_str.find(sub, start)
        if start == -1: return
        yield start
        start += len(sub) # use start += 1 to find overlapping matches
with open("/home/joe/bic_etl/general/cron/cron_file","r") as fin:
    for line in fin:
        if line[0:1] != "#"  and len(line) > 5:
           #print(line0)
#           print(line)
           crons.append(line.rstrip())
        line0=line
print(f"{len(crons)} Cron Jobs Found")

crons_all = []
ncrons=0
descriptions = []
for line in crons:
    line = line.strip(" ")
    spl = line.split(" ")
    crn=""
    print(line)
    mm = line.find("node")
    nj = line.find("java")
    npyth = line.find("python")
    nt = line.find("-t")
    np = line.find("-p")
    if (np > 0):
        a = list(find_all(line[np:],'\"'))
        # if len(a) > 0:
        #   p=line[np+a[0]:np+a[1]+1]
        # else:
        ss = line[np:].split()
        p=ss[1]
    elif mm > 0:
        pp = spl[6].split("/")
        p= pp[-1]
        
    elif nj > 0:  # java line
        pp = spl[7].split("/")
        p= pp[-1]
       
    elif npyth > 0:  # java line
        pp = spl[10].split("/")
        p= pp[-1]
       
    else:
            p=""
        
    if (nt > 0):
        ngt = line.find(">>")
        a = list(find_all(line[nt:ngt],'\"'))
        if len(a) > 0:
          t=line[nt+a[0]:nt+a[1]+1]
        else:
          ss = line[nt:].split()
          t=ss[1]
    else:
        t=""
            

    for val in spl[:5]:
        crn+= f"{val} "
    crn = crn.rstrip()
    if spl[5] == "node":
        pg = spl[6]
    else:
        pg=""
    if len(p) > 0:  #  There is a program listed
      
        pgs=pg.split("/")
       
    #    print(line)
        mo = 1
        yr=today.year
        mo=today.month
        
        
        try:
            it = CronSim(crn,datetime.strptime(f"{yr}-{mo}-01","%Y-%m-%d"))
            tmp={}
            tmp["line"]=line
            tmp["desc"] = it.explain()
            print("ME ",yr,mo,it.explain())
            descriptions.append(tmp)
            a = next(it) 
        
            while  a.month == mo:
         #       print(a.month,a.day,a.hour,a.minute,a.hour+a.minute/60)
         #       print("DAY ",calendar.day_name[a.weekday()])
          #      print(f"start:{a}   end:{a+timedelta(days=1)}")
                d = dict(Day=calendar.day_name[a.weekday()],Cron=line,T=t,TM=f"{a.hour}:{a.minute}",Task=p,Details=t,Program=pg,Start=a,End=a+timedelta(days=1),Time=a.hour+a.minute/60)
                crons_all.append(d)
            # print(f"pg: {pg} P:{p} T:{t}")    
                a = next(it)
            ncrons+=1 
        except Exception as err:
            print("Count not Process",crn)
            print(err)
            print(line)
    else:  # Not a node runnning a cim dataset... must be java or python
        print(line)
    #  print("--------")

print(f"{len(crons_all)}  Crons successfully mapped to time ranges")

30 Cron Jobs Found
50 2 * * * node /usr/local/cim/bic_etl/general/scripts/pull_and_setup.js 2>> $bic_etl_home/general/logs/cron.log
ME  2024 2 At 02:50 every day
58 2 * * * /usr/bin/java -jar /usr/local/cim/bic_etl/general/datasync/DataSync-1.8.2.jar -t LoadPreferences -c /usr/local/cim/bic_etl/general/datasync/config.json 2>> $bic_etl_home/general/logs/cron.log
ME  2024 2 At 02:58 every day
0 5 * * * PATH=$PATH:/home/giddensm/.local/bin PIPENV_PIPFILE=/usr/local/cim/bic_etl/general/metadata_updater/scripts/Pipfile pipenv run python /usr/local/cim/bic_etl/general/metadata_updater/scripts/metadata_updater.py -a 2>> $bic_etl_home/general/logs/cron.log
ME  2024 2 At 05:00 every day
10 3 * * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p boulder 2>> $bic_etl_home/general/logs/cron.log
ME  2024 2 At 03:10 every day
0 4 * * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p catalog 2>> $bic_etl_home/general/logs/cron.log
ME  2024 2 At 04:00 every day
5 6 * * 5 node /usr

In [8]:
ready4Xref = []
for ds in descriptions:
    print(ds["line"])
    st=ds["line"].find(r"-p")
    if st > 0:
       end = ds["line"][st+3:].find(r" ")
       group = ds["line"][st+2:st+3+end]
       print("Group: ",group)
       st = ds["line"].find(r"-t")
       ds["group"] = group
       if st > 0:
            st1 = ds["line"][st+3:].find('"')
            ed1 = ds["line"][st+3+st1+1:].find('"')
            dstitle = ds["line"][st+3+st1:st+3+st1+ed1+2]      
            print("title ",dstitle)
            ds["title"] = dstitle
       ready4Xref.append(ds) 
    print("-------------------------------------------------------\n")
    
            
            
      

50 2 * * * node /usr/local/cim/bic_etl/general/scripts/pull_and_setup.js 2>> $bic_etl_home/general/logs/cron.log
-------------------------------------------------------

58 2 * * * /usr/bin/java -jar /usr/local/cim/bic_etl/general/datasync/DataSync-1.8.2.jar -t LoadPreferences -c /usr/local/cim/bic_etl/general/datasync/config.json 2>> $bic_etl_home/general/logs/cron.log
-------------------------------------------------------

0 5 * * * PATH=$PATH:/home/giddensm/.local/bin PIPENV_PIPFILE=/usr/local/cim/bic_etl/general/metadata_updater/scripts/Pipfile pipenv run python /usr/local/cim/bic_etl/general/metadata_updater/scripts/metadata_updater.py -a 2>> $bic_etl_home/general/logs/cron.log
-------------------------------------------------------

10 3 * * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p boulder 2>> $bic_etl_home/general/logs/cron.log
Group:   boulder
-------------------------------------------------------

0 4 * * * node /usr/local/cim/bic_etl/general/scripts/bic_e

In [9]:
ready4Xref

[{'line': '10 3 * * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p boulder 2>> $bic_etl_home/general/logs/cron.log',
  'desc': 'At 03:10 every day',
  'group': ' boulder'},
 {'line': '0 4 * * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p catalog 2>> $bic_etl_home/general/logs/cron.log',
  'desc': 'At 04:00 every day',
  'group': ' catalog'},
 {'line': '5 6 * * 5 node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p cdos/business/nonprofit 2>> $bic_etl_home/general/logs/cron.log',
  'desc': 'At 06:05 on Friday',
  'group': ' cdos/business/nonprofit'},
 {'line': '5 5 * * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p cdos/business/nonprofit -t "Registration for Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado" -d /usr/local/cim/bic_etl/general/datasync/DataSync-1.9.1.jar 2>> $bic_etl_home/general/logs/cron.log',
  'desc': 'At 05:05 every day',
  'group': ' cdos/business/non

In [5]:

crons_all = []
ncrons=0
for line in crons:
    line = line.strip(" ")
    spl = line.split(" ")
    crn=""
#    print(line)
    mm = line.find("node")
    nj = line.find("java")
    npyth = line.find("python")
    nt = line.find("-t")
    np = line.find("-p")
    if (np > 0):
        a = list(find_all(line[np:],'\"'))
        # if len(a) > 0:
        #   p=line[np+a[0]:np+a[1]+1]
        # else:
        ss = line[np:].split()
        p=ss[1]
    elif mm > 0:
        pp = spl[6].split("/")
        p= pp[-1]
        print("P ",p)
    elif nj > 0:  # java line
        pp = spl[7].split("/")
        p= pp[-1]
        print("P ",p)
    elif npyth > 0:  # java line
        pp = spl[10].split("/")
        p= pp[-1]
        print("P ",p)
    else:
            p=""
        
    if (nt > 0):
        ngt = line.find(">>")
        a = list(find_all(line[nt:ngt],'\"'))
        if len(a) > 0:
          t=line[nt+a[0]:nt+a[1]+1]
        else:
          ss = line[nt:].split()
          t=ss[1]
    else:
        t=""
            
    
    for val in spl[:5]:
        crn+= f"{val} "
    crn = crn.rstrip()
    if spl[5] == "node":
        pg = spl[6]
    else:
        pg=""
    if len(p) > 0:  #  There is a program listed
      
        pgs=pg.split("/")
       
    #    print(line)
        mo = 11
        
        try:
            it = CronSim(crn,datetime.strptime("2023-11-01","%Y-%m-%d"))
            a = next(it) 
            
            while  a.month == mo:
           #     print("TIME TIME TIME TIME ",a.month,a.day,a.hour,a.minute,a.hour+a.minute/60)
          #      print(f"start:{a}   end:{a+timedelta(days=1)}")
             
                d = dict(Day=calendar.day_name[a.weekday()],Cron=line,T=t,TM=f"{a.hour}:{a.minute}",Task=p,Details=t,Program=pg,Start=a,End=a+timedelta(days=1),Time=a.hour+a.minute/60,Color=li[ncrons])
                crons_all.append(d)
              
            # print(f"pg: {pg} P:{p} T:{t}")    
                a = next(it)
            ncrons+=1 
        except:
            print("Count not Process",crn)
            print(line)
    else:  # Not a node runnning a cim dataset... must be java or python
        print(line)
    #  print("--------")

print(f"{len(crons_all)}  Crons successfully mapped to time ranges")

P  pull_and_setup.js
Count not Process 50 2 * * *
50 2 * * * node /usr/local/cim/bic_etl/general/scripts/pull_and_setup.js 2>> $bic_etl_home/general/logs/cron.log
P  DataSync-1.8.2.jar
Count not Process 58 2 * * *
58 2 * * * /usr/bin/java -jar /usr/local/cim/bic_etl/general/datasync/DataSync-1.8.2.jar -t LoadPreferences -c /usr/local/cim/bic_etl/general/datasync/config.json 2>> $bic_etl_home/general/logs/cron.log
P  metadata_updater.py
Count not Process 0 5 * * *
0 5 * * * PATH=$PATH:/home/giddensm/.local/bin PIPENV_PIPFILE=/usr/local/cim/bic_etl/general/metadata_updater/scripts/Pipfile pipenv run python /usr/local/cim/bic_etl/general/metadata_updater/scripts/metadata_updater.py -a 2>> $bic_etl_home/general/logs/cron.log
Count not Process 10 3 * * *
10 3 * * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p boulder 2>> $bic_etl_home/general/logs/cron.log
Count not Process 0 4 * * *
0 4 * * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p catalog 2>> $bic_etl_home/g

In [135]:
crons

['50 2 * * * node /usr/local/cim/bic_etl/general/scripts/pull_and_setup.js 2>> $bic_etl_home/general/logs/cron.log',
 '58 2 * * * /usr/bin/java -jar /usr/local/cim/bic_etl/general/datasync/DataSync-1.8.2.jar -t LoadPreferences -c /usr/local/cim/bic_etl/general/datasync/config.json 2>> $bic_etl_home/general/logs/cron.log',
 '0 5 * * * PATH=$PATH:/home/giddensm/.local/bin PIPENV_PIPFILE=/usr/local/cim/bic_etl/general/metadata_updater/scripts/Pipfile pipenv run python /usr/local/cim/bic_etl/general/metadata_updater/scripts/metadata_updater.py -a 2>> $bic_etl_home/general/logs/cron.log',
 '10 3 * * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p boulder 2>> $bic_etl_home/general/logs/cron.log',
 '0 4 * * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p catalog 2>> $bic_etl_home/general/logs/cron.log',
 '5 6 * * 5 node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p cdos/business/nonprofit 2>> $bic_etl_home/general/logs/cron.log',
 '5 5 * * * node /usr/local/cim/

In [6]:
len(crons_all)

0

In [10]:
df = pd.DataFrame(crons_all)

In [11]:
df.columns

Index(['Day', 'Cron', 'T', 'TM', 'Task', 'Details', 'Program', 'Start', 'End',
       'Time'],
      dtype='object')

In [12]:
a=df["Task"].value_counts().sort_index().to_dict()

In [13]:
for k in sorted(a.keys()):
    print(k)
    

DataSync-1.8.2.jar
boulder
catalog
cdor/regulations_liquor
cdor/retail_reports
cdor/revenue_marijuana
cdos/business/business
cdos/business/nonprofit
cdos/business/ucc
cdos/government
cdos/health
cdos/lobbyist
cdot/natural_resources
cdot/tops
cdot/transportation_infrastructure
cdot/transportation_road_attributes
ceo/useia
cleanup.js
dola/boundaries
dola/demographics
dola/special_districts
dora/regulations
metadata_updater.py
pull_and_setup.js


In [14]:
del a["pull_and_setup.js"]
del a["metadata_updater.py"]
del a["cleanup.js"]
del a["catalog"]
del a["boulder"]
del a["DataSync-1.8.2.jar"]

for key in sorted(a.keys()):
    print(key)


cdor/regulations_liquor
cdor/retail_reports
cdor/revenue_marijuana
cdos/business/business
cdos/business/nonprofit
cdos/business/ucc
cdos/government
cdos/health
cdos/lobbyist
cdot/natural_resources
cdot/tops
cdot/transportation_infrastructure
cdot/transportation_road_attributes
ceo/useia
dola/boundaries
dola/demographics
dola/special_districts
dora/regulations


In [16]:
def getXrefs():
    '''Reads the Inventory google sheet and gets the datasets title and cross-refs it to the Socrata 4x4 id.  Also
    gets the fields by 4x4 dataset id and by the title'''
    scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
             "https://www.googleapis.com/auth/drive.file",
                  "https://www.googleapis.com/auth/drive"]

    creds = ServiceAccountCredentials.from_json_keyfile_name('/home/joe/work/client_secret.json',
     scope)
    client = gspread.authorize(creds)

    gc = gspread.service_account("/home/joe/work/client_secret.json")
    # for gg in gc.list_spreadsheet_files():
    #      print("GGGGG ",gg)
    
    tracker = client.open('BIC Dataset Tracker').worksheet(
    'PublishedData')
   

    df = pd.DataFrame(tracker.get_all_records(head=3))

    return df
            
df = getXrefs()

/tmp/ipykernel_412/2438374747.py:24: DeprecationWarning: [Deprecated][in version 6.0.0]: client_factory will be replaced by gspread.http_client types
  df = getXrefs()


In [17]:
df.columns

Index(['Dataset Title', 'Short Description', 'Category', 'Keywords', 'Type',
       'License Type', 'Data Provider', 'Data Provided by', 'Source Link',
       'State Steward', 'Citation', 'Agency Program Page',
       'Agency Data Series Page', 'Business Contact and Phone',
       'Technical Contact and Phone', 'Data Source', 'Unit of Analysis',
       'Granularity Coverage', 'Geographic Extent and Division',
       'Collection Mode', 'Collection Methodology',
       'Data Collection Instrument', 'Date of Initial Dataset Creation',
       'Field Names, comma delimited', 'Oldest Record in Dataset',
       'Newest Record in Dataset', 'Long Description', 'Data Dictionary',
       'Additional Metadata', 'Technical Documentation',
       'Data Quality Certification',
       'Applicable Information Quality Guideline Designation',
       'Stewardship Plan', 'Collection Method', 'Horizontal Accuracy',
       'Horizontal Coordinate System', 'Update Schedule', 'Update Method',
       'Source Upd

In [22]:
dfS = df[['Socrata Link','Dataset Title','State Steward','Update Type','Update Schedule', 'Update Method','Source Update Schedule','CIM Updated', 'Days Since CIM Update',
       'cimAllData Updates', 'Complexity']]

In [23]:
for col in dfS.columns:
    print("Column ",col)
    print(dfS[col].value_counts())
    print("-----------------------------------------")

Column  Socrata Link
Socrata Link
cpwf-cznk    1
rifs-n6ib    1
c8jj-hcxj    1
82s5-cpkk    1
n55r-9hud    1
            ..
bynd-i2hj    1
k3gg-hhc8    1
x8tb-f3vh    1
2yhn-3dbj    1
6kn4-89kh    1
Name: count, Length: 399, dtype: int64
-----------------------------------------
Column  Dataset Title
Dataset Title
City of Denver Traffic Accidents                                                              1
Transparency Online Project (TOPS) - State Government Revenue and Expenditures in Colorado    1
Truck Station Electrification in Colorado 2014                                                1
GDP by Metropolitan Statistical Area                                                          1
Personal Consumption Expenditures                                                             1
                                                                                             ..
Consumer Price Index 2014                                                                     1
Retail Repor

In [20]:
dfAuto = dfS.loc[dfS["Update Type"].str.lower().str.contains("auto")]

In [24]:
dfS["State Steward"].value_counts()

State Steward
DOLA                         127
CDOT                          56
CDOS                          54
CEO                           32
CDOR                          24
City and county of Denver     22
                              17
CDLE                          10
CDPS                           5
Adams County                   5
Clear Creek County             4
Eagle County                   4
CDHE                           3
Boulder County                 3
DORA                           3
Garfield County                2
CDA                            2
TCHD                           2
BOCO                           2
Jefferson County               2
USGS                           2
#N/A                           2
Broomfield County              2
DPA                            1
DWR                            1
DHSEM                          1
CDPHE                          1
OITGIS                         1
CTO                            1
Town of Breckenridge         

### Get Dataset Titles from ETL Processes

Get datasets listed in the run_etl.json files from the github repo

In [26]:
def getTitles(string):
    titles=[]
    h = subprocess.check_output(string,shell=True)
    j = str(h).split("\\")
    for s in j:
       k = s.split(":")
       if len(k) > 1:
            title=k[1].strip()
            title=title.rstrip(",")
            print(title)
            titles.append(title)
    return titles
titlesG = {}
titles = {}
for key in sorted(a.keys()):
    string = f"grep -i title /home/joe/bic_etl/{key}/run_etl.json"
    
    titlesG[key] = getTitles(string)
    
    for title in titlesG[key]:
        print(title)
        titles[title] = key
       

"Liquor Permits for Special Events in Colorado"
"Liquor Compliance Check Statistics in Colorado"
"Liquor Licenses in Colorado"
"Recently Approved Liquor Licenses in Colorado"
"Recently Expired and Surrendered Liquor Licenses in Colorado"
"Sales Rooms in Colorado"
"Manufacturer Temporary Sales Room Permits in Colorado"
"Liquor Permits for Special Events in Colorado"
"Liquor Compliance Check Statistics in Colorado"
"Liquor Licenses in Colorado"
"Recently Approved Liquor Licenses in Colorado"
"Recently Expired and Surrendered Liquor Licenses in Colorado"
"Sales Rooms in Colorado"
"Manufacturer Temporary Sales Room Permits in Colorado"
"Retail Sales Tax Return History in Colorado"
"Retail Reports by City in Colorado"
"Retail Reports by County in Colorado"
"Retail Reports by Industry and City in Colorado"
"Retail Reports by Industry and County in Colorado"
"Retail Reports by Industry in Colorado"
"Retail Sales Tax Return History in Colorado"
"Retail Reports by City in Colorado"
"Retail Repo

In [27]:
titles

{'"Liquor Permits for Special Events in Colorado"': 'cdor/regulations_liquor',
 '"Liquor Compliance Check Statistics in Colorado"': 'cdor/regulations_liquor',
 '"Liquor Licenses in Colorado"': 'cdor/regulations_liquor',
 '"Recently Approved Liquor Licenses in Colorado"': 'cdor/regulations_liquor',
 '"Recently Expired and Surrendered Liquor Licenses in Colorado"': 'cdor/regulations_liquor',
 '"Sales Rooms in Colorado"': 'cdor/regulations_liquor',
 '"Manufacturer Temporary Sales Room Permits in Colorado"': 'cdor/regulations_liquor',
 '"Retail Sales Tax Return History in Colorado"': 'cdor/retail_reports',
 '"Retail Reports by City in Colorado"': 'cdor/retail_reports',
 '"Retail Reports by County in Colorado"': 'cdor/retail_reports',
 '"Retail Reports by Industry and City in Colorado"': 'cdor/retail_reports',
 '"Retail Reports by Industry and County in Colorado"': 'cdor/retail_reports',
 '"Retail Reports by Industry in Colorado"': 'cdor/retail_reports',
 '"Marijuana Sales by County in Colo

In [29]:
titleMap = {}
titleMap["CDOT Payroll"] = "CDOT Payroll Expenditures"
# titleMap[""] = ""
# titleMap[""] = ""
# titleMap[""] = ""
# titleMap[""] = ""

In [30]:
for title in titles.keys():
    tit=title
    title=title.strip('"')
    if title in titleMap:
        title = titleMap[title]
    tmp = dfS.loc[dfS["Dataset Title"].str.lower().str.strip() == title.lower()]
    if tmp.shape[0] < 1:
        print(f"{titles[tit]} :{title}:")
        tmp2 = df.loc[df["Dataset Title"].str.lower().str.strip() == title.lower()]
        print(tmp.shape,tmp2.shape)
        
        

cdor/retail_reports :Retail Sales Tax Return History in Colorado:
(0, 11) (0, 60)
cdor/revenue_marijuana :State Retail Marijuana Sales Tax Revenue by County in Colorado:
(0, 11) (0, 60)
cdor/revenue_marijuana :Marijuana Sales Revenue in Colorado:
(0, 11) (0, 60)
cdos/business/nonprofit :Registration for Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado:
(0, 11) (0, 60)
cdos/business/nonprofit :Charitable Organizations:
(0, 11) (0, 60)
cdos/business/nonprofit :Other State Solicitation of Charities:
(0, 11) (0, 60)
cdos/business/nonprofit :Other Names a Registered Entity Uses to Solicit Contributions:
(0, 11) (0, 60)
dola/demographics :Race Forecast in Colorado:
(0, 11) (0, 60)
dola/special_districts :Cemetery Districts in Colorado:
(0, 11) (0, 60)


In [131]:
#a = dfS.loc[dfS["Dataset Title"].str.lower().str.contains("cdot")]
df.loc[df["State Steward"].str.lower().str.contains("cdor")]

,Dataset Title,Short Description,Category,Keywords,Type,License Type,Data Provider,Data Provided by,Source Link,State Steward,...,Socrata Link,API 4x4,Web Display Coordinate System,Coordinate System Disclaimer,Related Datasets,Quarter of Gov FY Published,CIM Updated,Days Since CIM Update,cimAllData Updates,Complexity
19,Retail Reports by Industry and City in Colorado,"Number of returns, gross sales, retail sales, ...",Business,"bic, business, covid19, gocodecolorado, bic, d...",Business,Public Domain,CDOR,CDOR - Colorado Department of Revenue,https://www.colorado.gov/pacific/revenue/retai...,CDOR,...,k3gg-hhc8,k3gg-hhc8,,,,2,2024-01-08T11:28:32+0000,8,14,
20,Retail Reports by County in Colorado,"Number of returns, gross sales, retail sales, ...",Business,"bic, business, covid19, gocodecolorado, bic, d...",Business,Public Domain,CDOR,CDOR - Colorado Department of Revenue,https://www.colorado.gov/pacific/revenue/retai...,CDOR,...,x8tb-f3vh,x8tb-f3vh,,,,2,2024-01-08T11:28:36+0000,8,3305,
21,Retail Reports by City in Colorado,"Number of returns, gross sales, retail sales, ...",Business,"bic, business, covid19, gocodecolorado, bic, d...",Business,Public Domain,CDOR,CDOR - Colorado Department of Revenue,https://www.colorado.gov/pacific/revenue/retai...,CDOR,...,2yhn-3dbj,2yhn-3dbj,,,,2,2024-01-08T11:28:36+0000,8,92,
22,Retail Reports by Industry in Colorado,Monthly reports per industry in Colorado on sa...,Business,"bic, business, gocodecolorado, bic, retail, in...",Business,Public Domain,CDOR,CDOR - Colorado Department of Revenue,https://www.colorado.gov/pacific/revenue/retai...,CDOR,...,6kn4-89kh,6kn4-89kh,,,,2,2024-01-08T11:28:41+0000,8,#N/A,
24,Marijuana Tax and Fee Revenue in Colorado\r\n,These reports show monthly state sales tax (2....,Revenue,"bic, marijuana, sales, tax, state, county, rev...",Tax,Public Domain,CDOR,CDOR - Colorado Department of Revenue,https://www.colorado.gov/pacific/revenue/color...,CDOR,...,3sm5-jtur,3sm5-jtur,,,,3,2024-01-15T11:18:22+0000,1,#N/A,
25,Retail Marijuana Sales Tax Revenue by County i...,Revenues from the statewide tax on recreationa...,Revenue,"bic, cannabis, marijuana, business, revenue, t...",Tax,Public Domain,CDOR,CDOR - Colorado Department of Revenue,https://www.colorado.gov/pacific/revenue/color...,CDOR,...,v9m8-x8dh,v9m8-x8dh,,,,3,2024-01-15T11:18:18+0000,1,3305,
26,Marijuana Sales by County in Colorado,"Monthly marijuana sales in Colorado by County,...",Revenue,"bic, marijuana, sales, revenue, colorado, coun...",Sales,Public Domain,CDOR,CDOR - Colorado Department of Revenue,https://www.colorado.gov/pacific/revenue/color...,CDOR,...,j7a3-jgd3,j7a3-jgd3,,,,3,2024-01-15T11:18:24+0000,1,641,
27,Liquor Compliance Check Statistics in Colorado,"Business names, location, liquor license type ...",Regulations,"bic, gocodecolorado, colorado, dor, department...",Liquor Regulation,Public Domain,CDOR,CDOR - Colorado Department of Revenue,https://www.colorado.gov/pacific/enforcement/l...,CDOR,...,kapc-ib6e,ii5c-5549,,,Related to: Liquor Permits for Special Events ...,4,2023-09-22T10:40:55+0000,116,1724,
28,Recently Approved Liquor Licenses in Colorado,Names and locations of business with active li...,Regulations,"bic, gocodecolorado, colorado, dor, department...",Liquor Regulation,Public Domain,CDOR,CDOR - Colorado Department of Revenue,https://www.colorado.gov/pacific/enforcement/l...,CDOR,...,htyp-tqzh,htyp-tqzh,,,Related to: Liquor Permits for Special Events ...,4,2024-01-15T11:38:13+0000,1,599,
29,Liquor Licenses in Colorado,Names and locations of business with active li...,Regulations,"bic, gocodecolorado, colorado, dor, department...",Liquor Regulation,Public Domain,CDOR,CDOR - Colorado Department of Revenue,https://www.colorado.gov/pacific/enforcement/l...,CDOR,...,ier5-5ms2,ier5-5ms2,,,Related to: Liquor Permits for Special Events ...,4,2024-01-04T19:03:00+0000,11,1052,


In [ ]:
df